# Baselines Comparison & Ablation Study

This notebook implements baseline methods and ablation studies for comparing against the TGN detector.

In [1]:
import os
import sys
import json
import numpy as np
from pathlib import Path
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Configuration and Data

In [2]:
import yaml

# Load config
config_path = Path("../configs/default.yaml")
with open(config_path) as f:
    CONFIG = yaml.safe_load(f)

DATA_DIR = Path("../data/raw/llm_generated")
EXP_DIR = Path("../experiments/run_current_arch")

print(f"Data directory: {DATA_DIR}")
print(f"Experiment directory: {EXP_DIR}")


Data directory: ..\data\raw\llm_generated
Experiment directory: ..\experiments\run_current_arch


In [3]:
# Import dataset and model from main notebook (or copy definitions)
# For simplicity, we'll import from the src modules
sys.path.insert(0, str(Path("..").resolve()))

from src.data_generation.dataset import MemoryTrajectoryDataset, collate_trajectories
from src.models.tgn_detector import TGNDetector

## 2. Data Collection for Baselines

In [4]:
def collect_test_data(data_dir: str, config: dict):
    """Collect node features and labels from test set for baseline methods."""
    T = config["training"]["temporal_window"]
    embedding_dim = config["simulation"]["embedding_dim"]
    
    test_dataset = MemoryTrajectoryDataset(data_dir, split="test", temporal_window=T, embedding_dim=embedding_dim)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_trajectories)
    
    all_node_features = []
    all_node_labels = []
    all_graph_features = []
    all_graph_labels = []
    
    for batch in tqdm(test_loader, desc="Collecting test data"):
        graphs = batch["graphs"]
        graph_label = batch["graph_labels"].numpy()[0]
        
        # Get last snapshot
        last_graph = graphs[-1]
        node_features = last_graph.x.numpy()
        node_labels = last_graph.node_labels.numpy()
        
        all_node_features.append(node_features)
        all_node_labels.append(node_labels)
        
        # Graph-level: mean pool node features
        graph_feat = node_features.mean(axis=0)
        all_graph_features.append(graph_feat)
        all_graph_labels.append(graph_label)
    
    return {
        "node_features": np.vstack(all_node_features),
        "node_labels": np.concatenate(all_node_labels),
        "graph_features": np.vstack(all_graph_features),
        "graph_labels": np.array(all_graph_labels)
    }

test_data = collect_test_data(str(DATA_DIR), CONFIG)
print(f"Node features shape: {test_data['node_features'].shape}")
print(f"Graph features shape: {test_data['graph_features'].shape}")

Node features shape: (50506, 387)
Graph features shape: (330, 387)


## 3. Baseline 1: Isolation Forest

In [5]:
def evaluate_isolation_forest(test_data: dict):
    """Evaluate Isolation Forest baseline."""
    results = {}
    
    # Node-level
    iso_node = IsolationForest(contamination=0.1, random_state=42, n_jobs=-1)
    iso_node.fit(test_data["node_features"])
    node_scores = -iso_node.score_samples(test_data["node_features"])  # Higher = more anomalous
    node_scores = (node_scores - node_scores.min()) / (node_scores.max() - node_scores.min())  # Normalize
    
    if len(np.unique(test_data["node_labels"])) > 1:
        results["node_auroc"] = roc_auc_score(test_data["node_labels"], node_scores)
        results["node_ap"] = average_precision_score(test_data["node_labels"], node_scores)
    
    # Graph-level
    iso_graph = IsolationForest(contamination=0.4, random_state=42, n_jobs=-1)
    iso_graph.fit(test_data["graph_features"])
    graph_scores = -iso_graph.score_samples(test_data["graph_features"])
    graph_scores = (graph_scores - graph_scores.min()) / (graph_scores.max() - graph_scores.min())
    
    if len(np.unique(test_data["graph_labels"])) > 1:
        results["graph_auroc"] = roc_auc_score(test_data["graph_labels"], graph_scores)
        results["graph_f1"] = f1_score(test_data["graph_labels"], (graph_scores > 0.5).astype(int))
    
    return results

iso_results = evaluate_isolation_forest(test_data)
print("\n=== Isolation Forest Results ===")
for k, v in iso_results.items():
    print(f"  {k}: {v:.4f}")


=== Isolation Forest Results ===
  node_auroc: 0.5438
  node_ap: 0.0206
  graph_auroc: 0.5293
  graph_f1: 0.2111


## 4. Baseline 2: One-Class SVM

In [6]:
def evaluate_ocsvm(test_data: dict):
    """Evaluate One-Class SVM baseline."""
    results = {}
    
    # Node-level (subsample for speed)
    n_samples = min(5000, len(test_data["node_features"]))
    idx = np.random.choice(len(test_data["node_features"]), n_samples, replace=False)
    
    ocsvm = OneClassSVM(kernel='rbf', nu=0.1)
    ocsvm.fit(test_data["node_features"][idx])
    node_scores = -ocsvm.score_samples(test_data["node_features"])
    node_scores = (node_scores - node_scores.min()) / (node_scores.max() - node_scores.min() + 1e-8)
    
    if len(np.unique(test_data["node_labels"])) > 1:
        results["node_auroc"] = roc_auc_score(test_data["node_labels"], node_scores)
        results["node_ap"] = average_precision_score(test_data["node_labels"], node_scores)
    
    # Graph-level
    ocsvm_graph = OneClassSVM(kernel='rbf', nu=0.4)
    ocsvm_graph.fit(test_data["graph_features"])
    graph_scores = -ocsvm_graph.score_samples(test_data["graph_features"])
    graph_scores = (graph_scores - graph_scores.min()) / (graph_scores.max() - graph_scores.min() + 1e-8)
    
    if len(np.unique(test_data["graph_labels"])) > 1:
        results["graph_auroc"] = roc_auc_score(test_data["graph_labels"], graph_scores)
        results["graph_f1"] = f1_score(test_data["graph_labels"], (graph_scores > 0.5).astype(int))
    
    return results

ocsvm_results = evaluate_ocsvm(test_data)
print("\n=== One-Class SVM Results ===")
for k, v in ocsvm_results.items():
    print(f"  {k}: {v:.4f}")


=== One-Class SVM Results ===
  node_auroc: 0.5224
  node_ap: 0.0195
  graph_auroc: 0.2450
  graph_f1: 0.3375


## 5. Baseline 3: Static GNN (No Temporal Component)

In [7]:
class StaticGNNDetector(nn.Module):
    """Static GNN without temporal modeling - baseline for ablation."""
    
    def __init__(self, config: dict):
        super().__init__()
        model_cfg = config["model"]
        
        input_dim = model_cfg["node_feature_dim"] + 3
        gat_hidden = model_cfg["gat_hidden_dim"]
        gat_heads = model_cfg["gat_heads"]
        mlp_hidden = model_cfg["mlp_hidden_dim"]
        dropout = model_cfg["dropout"]
        
        # GAT layers
        self.gat1 = GATConv(input_dim, gat_hidden, heads=gat_heads, dropout=dropout, concat=True, edge_dim=1)
        self.gat2 = GATConv(gat_hidden * gat_heads, gat_hidden, heads=1, dropout=dropout, concat=False, edge_dim=1)
        
        # Scoring heads
        self.graph_head = nn.Sequential(
            nn.Linear(gat_hidden, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(mlp_hidden, 1)
        )
        self.node_head = nn.Sequential(
            nn.Linear(gat_hidden, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(mlp_hidden, 1)
        )
        self.dropout = dropout
    
    def forward(self, graph_sequence: List[Batch]):
        # Only use last snapshot (no temporal)
        g = graph_sequence[-1]
        x = g.x
        edge_index = g.edge_index
        batch_idx = g.batch
        edge_attr = g.edge_attr if hasattr(g, 'edge_attr') else None
        
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat1(x, edge_index, edge_attr=edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        node_emb = self.gat2(x, edge_index, edge_attr=edge_attr)
        
        graph_emb = global_mean_pool(node_emb, batch_idx)
        
        graph_scores = torch.sigmoid(self.graph_head(graph_emb)).squeeze(-1)
        node_scores = torch.sigmoid(self.node_head(node_emb)).squeeze(-1)
        
        return graph_scores, node_scores

In [8]:
def evaluate_static_gnn(config: dict, data_dir: str):
    """Train and evaluate Static GNN baseline.

    Trains on graph-level BCE loss plus a node-level BCE loss term (weight 0.5,
    matching beta in the proposed detector's joint objective, Eq. 3.20). An
    earlier version of this function trained on graph loss only, leaving
    node_head at random initialization for the whole run - its node metrics
    (node_auroc ~0.29) were an artifact of an untrained head, not a trained
    baseline result, confirmed by rerunning under a fixed seed and getting a
    wildly different value each time. Seed fixed at 42 for reproducibility.
    """
    import random
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    T = config["training"]["temporal_window"]
    embedding_dim = config["simulation"]["embedding_dim"]
    
    train_dataset = MemoryTrajectoryDataset(data_dir, split="train", temporal_window=T, embedding_dim=embedding_dim)
    test_dataset = MemoryTrajectoryDataset(data_dir, split="test", temporal_window=T, embedding_dim=embedding_dim)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_trajectories)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_trajectories)
    
    model = StaticGNNDetector(config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["training"]["learning_rate"])
    graph_loss_fn = nn.BCELoss()
    node_loss_fn = nn.BCELoss()
    node_loss_weight = 0.5

    # Quick training (5 epochs), graph loss + node loss
    for epoch in range(5):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for batch in train_loader:
            graphs = [g.to(device) for g in batch["graphs"]]
            graph_labels = batch["graph_labels"].to(device)
            
            graph_scores, node_scores = model(graphs)
            loss = graph_loss_fn(graph_scores, graph_labels)

            last_graph = graphs[-1]
            if hasattr(last_graph, "node_labels") and last_graph.node_labels is not None:
                node_labels = last_graph.node_labels.to(device).float()
                min_len = min(len(node_scores), len(node_labels))
                if min_len > 0:
                    node_loss = node_loss_fn(node_scores[:min_len], node_labels[:min_len])
                    loss = loss + node_loss_weight * node_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        print(f"  epoch {epoch + 1}/5 done, mean loss={epoch_loss / n_batches:.4f}")
    
    # Evaluate
    model.eval()
    all_graph_scores, all_graph_labels = [], []
    all_node_scores, all_node_labels = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            graphs = [g.to(device) for g in batch["graphs"]]
            graph_labels = batch["graph_labels"]
            
            graph_scores, node_scores = model(graphs)
            
            all_graph_scores.extend(graph_scores.cpu().numpy())
            all_graph_labels.extend(graph_labels.numpy())
            
            last_graph = graphs[-1]
            if hasattr(last_graph, "node_labels"):
                node_labels = last_graph.node_labels.cpu().numpy()
                ns = node_scores.cpu().numpy()
                min_len = min(len(ns), len(node_labels))
                all_node_scores.extend(ns[:min_len])
                all_node_labels.extend(node_labels[:min_len])
    
    results = {}
    graph_scores = np.array(all_graph_scores)
    graph_labels = np.array(all_graph_labels)
    
    if len(np.unique(graph_labels)) > 1:
        results["graph_auroc"] = roc_auc_score(graph_labels, graph_scores)
        results["graph_ap"] = average_precision_score(graph_labels, graph_scores)
        results["graph_f1"] = f1_score(graph_labels, (graph_scores > 0.5).astype(int))
    
    node_scores = np.array(all_node_scores)
    node_labels = np.array(all_node_labels)
    
    if len(np.unique(node_labels)) > 1:
        results["node_auroc"] = roc_auc_score(node_labels, node_scores)
        results["node_ap"] = average_precision_score(node_labels, node_scores)
    
    return model, results

static_gnn_model, static_gnn_results = evaluate_static_gnn(CONFIG, str(DATA_DIR))
print("\n=== Static GNN Results (node head now trained) ===")
for k, v in static_gnn_results.items():
    print(f"  {k}: {v:.4f}")

  epoch 1/5 done, mean loss=0.7698


  epoch 2/5 done, mean loss=0.6656


  epoch 3/5 done, mean loss=0.2730


  epoch 4/5 done, mean loss=0.0975


  epoch 5/5 done, mean loss=0.0719



=== Static GNN Results (node head now trained) ===
  graph_auroc: 0.9997
  graph_ap: 0.9991
  graph_f1: 0.9512
  node_auroc: 0.9733
  node_ap: 0.4998


### 5b. Static GNN - Partial-Visibility Probe (Early Detection)

Tests whether the newly-trained static baseline also fires on windows with only 1-2 visible poisoned records, using the same out-of-distribution probe sets and bucketing methodology as the full TGN detector's temporal probes (src/experiments/slow_attack_probe.py, Section 3.11). This settles whether temporal modelling contributes specifically to early detection (attacks in progress), independent of its near-parity with the static model on graph-level AUROC over the full test partition (which is dominated by fully-accumulated poisoned clusters, since the memory store never prunes).

In [9]:
from collections import defaultdict

def probe_partial_visibility(model, data_dir: str, temporal_window: int, label: str):
    """Buckets windows by number of poisoned records visible in the final snapshot,
    reports mean score and detection rate per bucket. Same methodology as
    src/experiments/slow_attack_probe.py, applied here to the static model."""
    with open(Path(data_dir) / "metadata.json") as f:
        metadata = json.load(f)
    all_seq_ids = [m["sequence_id"] for m in metadata]

    ds = MemoryTrajectoryDataset(data_dir, temporal_window=temporal_window, sequence_ids=all_seq_ids)
    print(f"Scoring {len(ds)} windows across {len(all_seq_ids)} sequences in {label}...")

    buckets = defaultdict(list)
    model.eval()
    with torch.no_grad():
        for i in range(len(ds)):
            item = ds[i]
            batch = collate_trajectories([item])
            graphs = [g.to(device) for g in batch["graphs"]]
            graph_scores, _ = model(graphs)
            score = graph_scores.squeeze().item()
            n_poisoned_visible = int(item["node_labels"].sum().item())
            buckets[n_poisoned_visible].append(score)

    print(f"\n[{label}] Detection rate (score >= 0.5) by number of poisoned records visible:")
    print(f"  {'n_poisoned':>10s}  {'n_windows':>9s}  {'mean_score':>10s}  {'detect_rate':>11s}")
    results = {}
    for n_poisoned in sorted(buckets.keys()):
        scores = np.array(buckets[n_poisoned])
        mean_score = float(scores.mean())
        detect_rate = float((scores >= 0.5).mean())
        results[str(n_poisoned)] = {"n_windows": len(scores), "mean_score": mean_score, "detect_rate": detect_rate}
        print(f"  {n_poisoned:>10d}  {len(scores):>9d}  {mean_score:>10.4f}  {detect_rate:>11.4f}")
    return results

single_record_static = probe_partial_visibility(
    static_gnn_model, "../data/raw/llm_generated_single_ood",
    CONFIG["training"]["temporal_window"], "single-record set (static GNN)")
slow_attack_static = probe_partial_visibility(
    static_gnn_model, "../data/raw/llm_generated_slow_ood",
    CONFIG["training"]["temporal_window"], "extended-spread set (static GNN)")

# Compare against the temporal detector's already-saved probe results for the same sets
with open("../experiments/single_record_probe_results.json") as f:
    single_record_tgn = json.load(f)
with open("../experiments/slow_attack_probe_results.json") as f:
    slow_attack_tgn = json.load(f)

print("\n=== Static GNN vs TGN Detector: partial-visibility comparison (bucket n=1, single-record set) ===")
print(f"  TGN Detector : mean_score={single_record_tgn['1']['mean_score']:.4f}  "
      f"detect_rate={single_record_tgn['1']['detect_rate']:.4f}  (n={single_record_tgn['1']['n_windows']})")
print(f"  Static GNN   : mean_score={single_record_static['1']['mean_score']:.4f}  "
      f"detect_rate={single_record_static['1']['detect_rate']:.4f}  (n={single_record_static['1']['n_windows']})")

# Save
partial_visibility_output = {
    "single_record_set": single_record_static,
    "extended_spread_set": slow_attack_static,
}
with open("../experiments/static_gnn_partial_visibility_results.json", "w") as f:
    json.dump(partial_visibility_output, f, indent=2)
print("\nSaved to ../experiments/static_gnn_partial_visibility_results.json")

Scoring 275 windows across 25 sequences in single-record set (static GNN)...



[single-record set (static GNN)] Detection rate (score >= 0.5) by number of poisoned records visible:
  n_poisoned  n_windows  mean_score  detect_rate
           0         52      0.0005       0.0000
           1        223      0.4116       0.3677
Scoring 420 windows across 20 sequences in extended-spread set (static GNN)...



[extended-spread set (static GNN)] Detection rate (score >= 0.5) by number of poisoned records visible:
  n_poisoned  n_windows  mean_score  detect_rate
           0          3      0.0001       0.0000
           2          3      0.9835       1.0000
           3         14      0.9676       1.0000
           4         12      0.9993       1.0000
           5         15      0.9986       1.0000
           6         11      0.9993       1.0000
           7         16      0.9992       1.0000
           8         29      0.9993       1.0000
           9         22      0.9993       1.0000
          10         16      0.9996       1.0000
          11         21      0.9997       1.0000
          12         22      0.9996       1.0000
          13         21      0.9998       1.0000
          14         28      0.9998       1.0000
          15         19      0.9997       1.0000
          16         17      0.9998       1.0000
          17         16      0.9997       1.0000
          18 

## 6. TGN Detector (Our Method)

In [10]:
def evaluate_tgn(config: dict, checkpoint_path: str, data_dir: str):
    """Evaluate trained TGN detector."""
    T = config["training"]["temporal_window"]
    embedding_dim = config["simulation"]["embedding_dim"]
    
    model = TGNDetector(config).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    model.eval()
    
    test_dataset = MemoryTrajectoryDataset(data_dir, split="test", temporal_window=T, embedding_dim=embedding_dim)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_trajectories)
    
    all_graph_scores, all_graph_labels = [], []
    all_node_scores, all_node_labels = [], []
    
    with torch.no_grad():
        model.reset_memory()
        for batch in test_loader:
            graphs = [g.to(device) for g in batch["graphs"]]
            graph_labels = batch["graph_labels"]
            
            graph_scores, node_scores = model(graphs)
            
            all_graph_scores.extend(graph_scores.cpu().numpy())
            all_graph_labels.extend(graph_labels.numpy())
            
            last_graph = graphs[-1]
            if hasattr(last_graph, "node_labels"):
                node_labels = last_graph.node_labels.cpu().numpy()
                ns = node_scores.cpu().numpy()
                min_len = min(len(ns), len(node_labels))
                all_node_scores.extend(ns[:min_len])
                all_node_labels.extend(node_labels[:min_len])
    
    results = {}
    graph_scores = np.array(all_graph_scores)
    graph_labels = np.array(all_graph_labels)
    
    if len(np.unique(graph_labels)) > 1:
        results["graph_auroc"] = roc_auc_score(graph_labels, graph_scores)
        results["graph_ap"] = average_precision_score(graph_labels, graph_scores)
        results["graph_f1"] = f1_score(graph_labels, (graph_scores > 0.5).astype(int))
    
    node_scores = np.array(all_node_scores)
    node_labels = np.array(all_node_labels)
    
    if len(np.unique(node_labels)) > 1:
        results["node_auroc"] = roc_auc_score(node_labels, node_scores)
        results["node_ap"] = average_precision_score(node_labels, node_scores)
    
    return results

checkpoint_path = EXP_DIR / "phase2_best.pt"
tgn_results = evaluate_tgn(CONFIG, str(checkpoint_path), str(DATA_DIR))
print("\n=== TGN Detector Results ===")
for k, v in tgn_results.items():
    print(f"  {k}: {v:.4f}")


=== TGN Detector Results ===
  graph_auroc: 1.0000
  graph_ap: 1.0000
  graph_f1: 0.9822
  node_auroc: 0.9737
  node_ap: 0.6343


## 7. Results Summary Table

In [11]:
import pandas as pd

# Compile results
all_results = {
    "Isolation Forest": iso_results,
    "One-Class SVM": ocsvm_results,
    "Static GNN": static_gnn_results,
    "TGN Detector (Ours)": tgn_results
}

# Create DataFrame
metrics = ["graph_auroc", "graph_f1", "node_auroc", "node_ap"]
df = pd.DataFrame(index=all_results.keys(), columns=metrics)

for method, results in all_results.items():
    for metric in metrics:
        df.loc[method, metric] = results.get(metric, "N/A")

print("\n" + "="*60)
print("BASELINES COMPARISON")
print("="*60)
print(df.to_string())
print("="*60)


BASELINES COMPARISON
                    graph_auroc  graph_f1 node_auroc   node_ap
Isolation Forest        0.52926  0.211111   0.543805  0.020611
One-Class SVM          0.244996  0.337531   0.522377  0.019511
Static GNN             0.999666   0.95122   0.973346  0.499837
TGN Detector (Ours)         1.0  0.982249    0.97374  0.634273


## 8. Ablation Study

In [12]:
ablation_results = {}

# Full model (already computed)
ablation_results["Full Model"] = tgn_results

# Without temporal (Static GNN, node head now trained comparably - see 5b)
ablation_results["Without Temporal"] = static_gnn_results

print("\n" + "="*60)
print("ABLATION STUDY")
print("="*60)

ablation_metrics = ["graph_auroc", "graph_ap", "graph_f1", "node_auroc", "node_ap"]
ablation_df = pd.DataFrame(index=ablation_results.keys(), columns=ablation_metrics)
for method, results in ablation_results.items():
    for metric in ablation_metrics:
        ablation_df.loc[method, metric] = results.get(metric, "N/A")

print(ablation_df.to_string())
print("="*60)


ABLATION STUDY
                 graph_auroc  graph_ap  graph_f1 node_auroc   node_ap
Full Model               1.0       1.0  0.982249    0.97374  0.634273
Without Temporal    0.999666  0.999106   0.95122   0.973346  0.499837


## 9. Save Results

In [13]:
# Save results to JSON
output = {
    "baselines": {k: {m: float(v) if isinstance(v, (int, float, np.floating)) else v 
                      for m, v in res.items()} 
                  for k, res in all_results.items()},
    "ablation": {k: {m: float(v) if isinstance(v, (int, float, np.floating)) else v 
                     for m, v in res.items()} 
                 for k, res in ablation_results.items()}
}

output_path = EXP_DIR / "baselines_comparison.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {output_path}")

Results saved to ..\experiments\run_current_arch\baselines_comparison.json
